# German Credit Analysis

In [ ]:
import pandas as pd
import numpy as np
import torch

from explainers.model import Model
from explainers.nondifferentiable import DCENonDifferentiable

from dataset.cardio import CardioData
from dataset.german_credit import GermanCreditData
from dataset.hotel_booking import HotelBookingData

from explainers.strategies.genetic import GeneticStrategy
from explainers.strategies.monte_carlo import MonteCarloStrategy
from explainers.strategies.simulated_annealing import SimulatedAnnealingStrategy
from explainers.strategies.bayesian import BayesianStrategy
from explainers.strategies.covariance_matrix_adaptation_evolution import CovarianceMatrixAdaptationEvolutionStrategy
from explainers.strategies.differential_evolution import DifferentialEvolutionStrategy
from explainers.strategies.particle_swarm_optimization import ParticleSwarmOptimizationStrategy

pd.set_option('display.max_columns', None)

%reload_ext autoreload
%autoreload 2

In [ ]:
seed = 42

np.random.seed(seed)
torch.manual_seed(seed)

## Dataset

In [ ]:
# data = CardioData(regenerate=False, seed = seed)

In [ ]:

data = GermanCreditData(regenerate=False, seed = seed)

In [ ]:
# data = HotelBookingData(regenerate=False, seed = seed)

In [ ]:
df_explain = data.get_df_explain(sample_num=50)
X_train, _, y_train, _ = data.get_train_test()

## Model

In [ ]:
# === MLP ===
import os
from models.mlp import BlackBoxModel
import torch
import torch.nn as nn
import torch.optim as optim

reproduce = False

checkpoint_path = "checkpoints/mlp_credit.pth"

X_train, _, y_train, _ = data.get_train_test()
input_dim = X_train.shape[1]
model_raw = BlackBoxModel(input_dim=input_dim)

if reproduce and os.path.exists(checkpoint_path):
    print("📦 Loading model from checkpoint...")
    model_raw.load_state_dict(torch.load(checkpoint_path))
    model_raw.eval()
else:
    print("🔁 Training model from scratch...")
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model_raw.parameters(), lr=0.01)

    X_tensor = torch.FloatTensor(X_train.values)
    y_tensor = torch.FloatTensor(y_train.values).view(-1, 1)

    for _ in range(300):
        pred = model_raw(X_tensor)
        loss = criterion(pred, y_tensor)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(model_raw.state_dict(), checkpoint_path)
    print("✅ Model saved to", checkpoint_path)

model = Model(model=model_raw, backend="pytorch", data=data) # "sklearn"  "pytorch"  "lightgbm"  "xgboost"

In [ ]:
# === LR ===
# from models.lr import LRModel
# model_raw = LRModel()
# model_raw.fit(X_train, y_train)
# model = Model(model=model_raw, backend="sklearn", data=data)


# === RBF ===
# from models.rbf import RBFNet
# model_raw = RBFNet(input_dim=X_train.shape[1], hidden_dim=X_train.shape[1])
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(model_raw.parameters(), lr=0.01)
# X_tensor = torch.FloatTensor(X_train.values)
# y_tensor = torch.FloatTensor(y_train.values).view(-1, 1)
# for _ in range(300):
#     pred = model_raw(X_tensor)
#     loss = criterion(pred, y_tensor)
#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()
# model = Model(model=model_raw, backend="pytorch", data=data)


# # === Random Forest ===
# from sklearn.ensemble import RandomForestClassifier
# model_raw = RandomForestClassifier(n_estimators=100, random_state=42)
# model_raw.fit(X_train, y_train)
# model = Model(model=model_raw, backend="sklearn", data=data)


# # === LightGBM ===
# from lightgbm import LGBMClassifier
# model_raw = LGBMClassifier(n_estimators=100, random_state=42)
# model_raw.fit(X_train, y_train)
# model = Model(model=model_raw, backend="lightgbm", data=data)


# === XGBoost ===
# from xgboost import XGBClassifier
# model_raw = XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")
# model_raw.fit(X_train, y_train)
# model = Model(model=model_raw, backend="xgboost", data=data)

In [ ]:
# model = Model(model=model_raw, backend="sklearn", data=data) # "sklearn"  "pytorch"  "lightgbm"  "xgboost"

## DCE for Non-Differentiable Models

In [ ]:
explainer = DCENonDifferentiable(model, data) 

strategy = MonteCarloStrategy(explainer, random_state = seed)

# strategy = GeneticStrategy(
#     explainer,     
#     crossover_prob=0.8,
#     gene_swap_prob=0.5,
#     mutation_prob_cat=0.3,
#     mutation_prob_cont=0.8,
#     mutation_noise_scale=0.1, 
#     random_state = seed
# )

# strategy = SimulatedAnnealingStrategy(
#     explainer,
#     T0=1.5,
#     T_final=0.01,
#     random_state = seed
# )

# strategy = BayesianStrategy(explainer, random_state = seed)

# strategy = DifferentialEvolutionStrategy(
#     explainer,
#     F=0.5, 
#     CR=0.9,
#     random_state = seed
# )

# strategy = CovarianceMatrixAdaptationEvolutionStrategy(
#     explainer,
#     population_size=10,
#     sigma_decay=0.9,
#     random_state = seed
# )

# strategy = ParticleSwarmOptimizationStrategy(
#     explainer,
#     swarm_size=30,
#     w=0.5,
#     c1=1.0,
#     c2=1.0,
#     random_state = seed
# )

df_cf = explainer.explain(
    df_factual=df_explain,
    X_init=None,
    n_proj=5,
    delta=0.1,
    U_1=0.5,
    U_2=0.3,
    l=0.2,
    r=1.0,  
    strategy=strategy,
    max_iter=30,
    top_k=1,
    callback="full"  # mode = "off" | "final_only" | "full"
)